In [ ]:
from jaxtyping import Array, Float, Scalar, Key
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
from matplotlib import pyplot as plt
from tqdm import tqdm
jax.config.update("jax_enable_x64", True)

from src import lisa, networks

In [ ]:
SEED = 0
N_SOURCES = 2
T_OBS = lisa.MONTH_s
TRAIN_STEPS = 1000

In [ ]:
u = jr.uniform(jr.key(SEED), shape=(N_SOURCES, 8))
params = lisa.prior_inverse_cdf(u)
signal = lisa.clean_signal(params, t_obs=T_OBS)
noise = lisa.sample_noise(jr.key(SEED+1), t_obs=T_OBS)
datastream = signal + noise
print(signal.shape, noise.shape)

n_samples = int(T_OBS / lisa.SAMPLING_STEP_s)
freqs = jnp.fft.rfftfreq(n_samples, lisa.SAMPLING_STEP_s)

plt.figure(figsize=(12, 8))
for j, channel in enumerate("AET"):
    plt.subplot(311 + j)
    plt.title(f"channel {channel}")
    plt.loglog(freqs, jnp.abs(datastream[:, j]), alpha=0.3, label="datastream")
    plt.loglog(freqs, jnp.abs(signal[:, j]), alpha=0.3, label="signal")
    plt.xlim(params[:, 0].min(axis=0) * 0.9, params[:, 0].max(axis=0) * 1.1)
    plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import blackjax

def loglik(pars):
    pars_model = pars.reshape(N_SOURCES, 8)
    model = lisa.clean_signal(pars_model, t_obs=T_OBS)
    res = datastream - model
    toret = 0.
    for i, ch in enumerate("AET"):
        PSD = lisa.noise_psd(ch)
        toret += jnp.sum(jnp.abs(res[1:, i]) ** 2 / PSD(freqs[1:]))
    return -0.5 * toret

ndim = params.flatten().shape[0]
nwalkers = ndim * 4
n_samples = 1000
params_MCMC = params.flatten()
p0 = params_MCMC * (1 + 1e-4 * jr.normal(jr.key(SEED + 2), shape=(nwalkers, ndim)))

# Proposal scale: 0.1% of each parameter value (gradient-free, no AD needed)
sigma = jnp.abs(params_MCMC) * 1e-3
rwm = blackjax.normal_random_walk(loglik, sigma)

@jax.jit
def run_sampling(key, positions):
    init_states = jax.vmap(rwm.init)(positions)

    def one_step(states, key):
        keys = jr.split(key, nwalkers)
        new_states, infos = jax.vmap(rwm.step)(keys, states)
        return new_states, (new_states.position, infos.acceptance_rate)

    _, (chain, acc) = jax.lax.scan(one_step, init_states, jr.split(key, n_samples))
    return chain, acc  # (n_samples, nwalkers, ndim), (n_samples, nwalkers)

print("Compiling and running...")
chain, acc = run_sampling(jr.key(SEED + 3), p0)
chain.block_until_ready()
print(f"Done. Chain shape: {chain.shape}")
print(f"Mean acceptance rate: {acc.mean():.2f}")


In [ ]:
import corner
import numpy as np

param_names = [r"$f_0$", r"$\dot{f}$", r"$A$", r"$\alpha$", r"$\delta$", r"$\psi$", r"$\iota$", r"$\phi_0$"]
labels = [f"{n} [{i+1}]" for i in range(N_SOURCES) for n in param_names]

# Flatten walkers into samples: (n_samples, nwalkers, ndim) -> (n_samples * nwalkers, ndim)
samples = np.array(chain).reshape(-1, ndim)

fig = corner.corner(
    samples,
    labels=labels,
    truths=np.array(params_MCMC),
    truth_color="tab:orange",
    show_titles=True,
    title_kwargs={"fontsize": 8},
    label_kwargs={"fontsize": 8},
)
plt.show()


In [ ]:
x, dx, t, y = lisa.get_train_batch(
    jr.key(0), batch_size=2, n_sources=N_SOURCES, t_obs=T_OBS
)
flow = networks.MMDiT(
    x_dim=x.shape[-1],
    y_dim=y.shape[-1],
    hidden_dim=128,
    num_blocks=4,
    num_heads=4,
    key=jr.key(0),
)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(3e-4),
)
opt_state = optimizer.init(eqx.filter(flow, eqx.is_array))


@eqx.filter_jit
def train_step(flow, opt_state, batch):
    xt, dx, t, y = batch

    def loss_fn(flow):
        pred = jax.vmap(flow)(xt, y, t)
        return jnp.mean((pred - dx) ** 2)

    loss, grads = eqx.filter_value_and_grad(loss_fn)(flow)
    updates, opt_state = optimizer.update(
        grads, opt_state, eqx.filter(flow, eqx.is_array)
    )
    flow = eqx.apply_updates(flow, updates)
    return flow, opt_state, loss


for key in (pbar := tqdm(jr.split(jr.key(SEED), TRAIN_STEPS))):
    batch = lisa.get_train_batch(key, batch_size=32, n_sources=N_SOURCES, t_obs=T_OBS)
    flow, opt_state, loss = train_step(flow, opt_state, batch)
    pbar.set_postfix(loss=loss.item())